# Byte-Level BPE Tokenizer

Our last tokenizer is the byte-level BPE, which is the closest to what will be implemented in the real tokenizer used for the LLM.

---

The process isn't much different from standard BPE:

raw text -> UTF-8 bytes -> byte-level base tokens -> BPE merges -> token IDs -> decode back to text

In [ ]:
corpus = (
# [Chorus: Childish Gambino]
"""
See now, I don't wanna see an era, an era, an era
See now, I just wanna live forever, and ever
Maybe it's the last night
Maybe it's the last night
"""

# [Verse 1: Childish Gambino]
"""
When the world was discerned with this and that
I was young, didn't know to hold it back
Now we here and the world is something else
We could leave any day and call for help
We were gods, nobody was above me (Woo)
That "A" on my chest like adultery (Yeah)
That "A" on my chest, put your fist up (Yeah)
That "A" on my chest like a chipmunk
Alvin, Theodore
Breathe in, breathe out, mi amor
Breathe in, breathe out, never mind
It was time, number nine
L-L-Lies, feline with a death wish
Eatin' right, didn't help shit
No name on the guest list
Hold on, we were destined
"""

# [Chorus: Azealia Banks & Childish Gambino]
"""
See now, I don't wanna see an era (No) An era (No) an era
See now, I just wanna live forever and ever
Maybe it’s the last night (You and me)
Maybe it’s the last night (Fireworks)
Maybe it’s the last night (Oh, oh, all the stars)
Maybe it’s the last night (Supermoon)
Maybe it's the last night
"""

# [Bridge: Azealia Banks]
"""
All that I have
All I don't have's the future
Take on my soul
All night
"""

# [Verse 2: Childish Gambino]
"""
Yeah, break free from all of the insides
The godless denied us
But we don't give a damn 'bout the next day
We were never here, never know if the world change
Common sense, the consequences
Retweet the truth, then regret the mentions
We are the wireless survivors of things gone
Slaves to the unnamed, never live long
Live and die by the line, "Never know fear"
Even I won't survive, is it unfair?
Is it unfair? 'Cause I don't care
When I step on that ant on the grass
Progress only thing that'll last (Last, last)
3005, the year that we fear only God will survive
To be alone is alive
"""

# [Chorus: Childish Gambino, Azealia Banks & Both]
"""
See now, I don’t wanna see an era, an era, an era
See now, I just wanna live forever and ever
Maybe it’s the last night
Maybe it’s the last night
Maybe it’s the last night
Maybe it’s the last night
Maybe it’s the last night
See now, I don’t wanna see an era, an era, an era
See now, I just wanna live forever and ever
Maybe it’s the last night
Maybe it’s the last night
Maybe it’s the last night
Maybe it’s the last night
Maybe it’s the last night
"""

# [Breakdown: Childish Gambino]
"""
She got that body, oh, me, oh, my
Send them pics to my phone, GPOY
She got that body, oh, me, oh, my
Send them pics to my phone, GPOY
GPOY, GPOY
GPOY, GPOY
GPOY, GPOY
Send them pics to my phone, GPOY
"""

# [Outro: Childish Gambino]
"""
So fly
Girl, you know you're so fly, so fly
Girl, I know I miss you
Girl, you know that I miss you
So fly
Girl, you know you're so fly, so fly
Girl, I know I miss you (I do)
Girl, you know that I miss you (I do, ooh)
You're so fly
Girl, you know you're so fly, so fly (I do, I do)
Girl, I know I miss you (Ah, I do)
Girl, you know that I miss you (I do, I do)
You're so fly
Girl, you know you're so fly, so fly (Ooh, ooh)
Girl, I know I miss you (Ooh)
Girl, you know that I miss you (Ooh, oh)
"""
# [Produced by Childish Gambino, Ludwig Göransson & Stefan Ponce]
)

print(corpus)

Unlike stardard (character-level) BPE, the base vocabulary consists of 256 possible byte values (0-255), this allows **any** UTF-8 character to be represented.

In [ ]:
text = "Child Gambino"

encoded_bytes = text.encode("utf-8") # default is utf-8 but we'll explicitly state it

print(encoded_bytes)
print(list(encoded_bytes))

ASCII characters corresponds to one byte, but special characters such as emojis take up multiple.

In [ ]:
emoji = "🤑"

encoded_bytes = emoji.encode("utf-8")

print(list(encoded_bytes))
print("Length of byte sequence for emoji: ", len(encoded_bytes))


In [ ]:
print("\nLength of character sequence for corpus: ", len(list(corpus)))
print("Length of word sequence for corpus: ", len(corpus.split()))

In [ ]:
byte_sequence = list(corpus.encode("utf-8"))

print(byte_sequence)
print("Length of byte sequence for corpus: ", len(byte_sequence))

The `merge_pair()` function is basically the same as the one for BPE, later functions require different implementation to account for merging bytes instead of characters/subwords.

In [ ]:
def merge_pair(tokens, pair, new_token_id):
    merged_tokens = []

    left, right = pair
    i = 0

    # Replace matching pairs into new token
    while i < len(tokens):
        if i < len(tokens) - 1 and tokens[i] == left and tokens[i + 1] == right:
            merged_tokens.append(new_token_id)
            i += 2
        else:
            merged_tokens.append(tokens[i])
            i += 1

    return merged_tokens

Another subtlety to notice is that `merge_pair()` is that it's **non-overlapping**, once a pair is successfully merged, `i` increases by 2. No token get's to participate in a second merge during one pass.

In [ ]:
tokens = [100, 101, 100, 101, 102]

merged = merge_pair(
    tokens,
    pair=(100, 101),
    new_token_id=256
)

print(merged)

`count_pairs()` is also basically the same the one in BPE, except we don't iterative though each word.

In [ ]:
from collections import Counter

def count_pairs(tokens):
    pair_count = Counter() # dictionary of "(part1, part2):count" pairs

    for i in range(len(tokens) - 1): # iterative tokens that have a next
        pair = tokens[i], tokens[i + 1] # pair them
        pair_count[pair] += 1 # increment count for that pair

    return pair_count

In [ ]:
tokens = [100, 101, 100, 101, 102, 101, 100]

pair_freq = count_pairs(tokens)

print(pair_freq)

## Main Byte BPE Implementation

The overall BBPE algorithm stayed pretty similar to the character-level BPE's code:

count adjacent token pairs -> find most frequent pair -> merge apply to corpus -> repeat

The main thing to keep in mind is what the tokenizer (vocabulary) starts with. BBPE starts with the 256 byte representations (0-255). Because these tokens are integers, we can't simply add the token IDs together to merge them, like how we did for character/subwords.

---

Instead, every learned merge gets a new token ID starting at `256` and
````python
vocab[new_token_id] = vocab[left] + vocab[right]
````
stores the merged parts separately.

---

````python
best_pair, count = pair_counts.most_common(1)[0]
if count < 2:
    break
````
also avoids "useless" merges if we merge every higher frequency pair before the merge limit.

In [ ]:
def train_byte_bpe(text, num_merges):
    # Convert to bytes
    tokens = list(text.encode("utf-8"))

    # Initial vocabulary of 256 possible bytes, ID 0-255
    vocab = {
        i: bytes([i])
        for i in range(256)
    }

    # Store learned merge rules
    merges = []

    # First available ID after the 256 base byte tokens
    next_token_id = 256

    for _ in range(num_merges):
        pair_counts = count_pairs(tokens)

        # Nothing left to merge
        if not pair_counts:
            break

        # Find most frequent adjacent token pair
        best_pair = pair_counts.most_common(1)[0][0]

        # Avoid merging one-off tokens
        best_pair, count = pair_counts.most_common(1)[0]
        if count < 2:
            break

        # Assign token ID for this merge
        new_token_id = next_token_id
        next_token_id += 1 # increment available ID

        # Store learned merge rule
        merges.append((best_pair, new_token_id))

        # Store what this new token represents as bytes
        left, right = best_pair
        vocab[new_token_id] = vocab[left] + vocab[right] # leftright:id

        # Apply the new merge to corpus
        tokens = merge_pair(
            tokens,
            best_pair,
            new_token_id
        )

    return tokens, merges, vocab

In [ ]:
trained_tokens, merges, vocab = train_byte_bpe(
    corpus,
    num_merges=50
)

print("Original byte count:", len(corpus.encode("utf-8")))
print("Token count after BPE:", len(trained_tokens))

print("\nNew Learned merges:")
for pair, token_id in merges:
    print(
        pair,
        "->",
        token_id,
        "->",
        vocab[token_id]
    )

A good thing about BBPE is that we don't explicitly need an unknown token, as every unknown falls back to byte IDs 0-255.

In [ ]:
id_to_token = vocab

# Reverse vocab
token_to_id = {
    token: token_id
    for token_id, token in id_to_token.items()
}

In [ ]:
def encode_bbpe(text, merges):
    tokens = list(text.encode("utf-8"))

    # Merge applicable pairs in byte sequence
    for pair, token_id in merges:
        tokens = merge_pair(
            tokens,
            pair,
            token_id
        )

    return tokens

def decode_bbpe(token_ids, id_to_token):
    byte_sequence = b"".join(
        id_to_token[token_id] # get byte
        for token_id in token_ids # for each id in sequence
    )

    return byte_sequence.decode("utf-8")

In [ ]:
text = "Maybe it's the last night"

encoded = encode_bbpe(
    text,
    merges
)

decoded = decode_bbpe(
    encoded,
    id_to_token
)

print("Original:", text)

print("\nEncoded:", encoded)
print("Decoded:", decoded)

print("\nToken count:", len(encoded))
print("Byte count:", len(text.encode("utf-8")))

Unknown text gets handled gracefully, but naturally are made up of a longer sequence.

In [ ]:
unk_text = "Donald Glover 🤑"

unk_encoded = encode_bbpe(unk_text, merges)
unk_decoded = decode_bbpe(unk_encoded, id_to_token)

print("Original:", unk_text)
print("Encoded:", unk_encoded)
print("Decoded:", unk_decoded)